# Taller I · Apache Spark: RDDs, DataFrames y Spark SQL

| | |
|---|---|
| **Curso** | MCDI502 · Gestión de Datos y Tecnologías Big Data |
| **Programa** | Magíster en Ciencia de Datos · Universidad Andrés Bello |
| **Evaluación** | Sumativa 2 · Taller I |
| **Grupo** | 6 |
| **Integrantes** | Fernanda Ovalle Román<br>Sebastián Cajales Cid<br>César Lorca Bacián<br>Jorge Álvarez Ossandón |
| **Repositorio** | [github.com/jaossandon/mcdi502-s2-spark](https://github.com/jaossandon/mcdi502-s2-spark) |

**Datos:** registros históricos de los Juegos Olímpicos (1896–2016) en seis archivos: deportistas (dos partes), equipos, eventos, juegos y resultados. La tabla de **resultados** es el punto de conexión que relaciona deportistas, equipos, juegos y eventos.

## Índice

| # | Sección | Indicador | Puntaje |
|---|---|---|---|
| 1 | Configuración inicial del entorno | – | 6 |
| 2 | RDDs: creación y unión | ID 3.1 | 18 |
| 3 | RDDs: transformaciones | ID 3.2 | 12 |
| 4 | DataFrames: creación e integración | ID 3.1 | 18 |
| 5 | Paralelismo | ID 3.4 | 9 |
| 6 | Inspección del DataFrame | ID 3.1 | 6 |
| 7 | Columnas calculadas | ID 3.3 | 12 |
| 8 | Agregaciones requeridas | ID 3.3 | 21 |
| 9 | Conclusiones y trazabilidad con la rúbrica | – | – |

### Cómo leer este notebook
Cada paso tiene tres partes:
1. **🎯 Qué haremos**: objetivo, decisión técnica y criterio de la rúbrica que cubre.
2. **Código**, comentado.
3. **✅ Resultado**: qué se obtuvo y cómo se interpreta.

---
# 1. Configuración inicial del entorno

Apache Spark está escrito en Scala y se ejecuta sobre la **JVM (Java Virtual Machine)**. Para usarlo desde Python se necesitan dos piezas: **Java**, que ejecuta el motor de Spark, y **PySpark**, la interfaz de Python que le envía las instrucciones.

## 1.1 Instalación de Java y Apache Spark

**🎯 Qué haremos**
- Detectar si el notebook se ejecuta en **Google Colab** o en un equipo local.
- En Colab: instalar **OpenJDK 17** y **PySpark 4.0.1** (el paquete ya incluye los binarios de Spark).
- En local: usar el entorno virtual del proyecto, donde ya están instalados.

**Decisión técnica:** se fijan versiones exactas (PySpark 4.0.1 y Java 17) para que el notebook sea **reproducible**. Spark 4.0 es la primera versión compatible con Python 3.13 y requiere Java 17 o superior.

In [3]:
# 1.1 Instalación de Java y Apache Spark
import sys, subprocess

EN_COLAB = "google.colab" in sys.modules
print("¿Ejecutando en Google Colab?:", EN_COLAB)

if EN_COLAB:
    # Java 17 sin interfaz gráfica (más liviano)
    subprocess.run("apt-get update -qq > /dev/null && "
                   "apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    # PySpark con la misma versión que el entorno local
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.1"], check=True)

# Evidencia: versiones disponibles
import pyspark
print("Versión de Python :", sys.version.split()[0])
print("Versión de PySpark:", pyspark.__version__)

¿Ejecutando en Google Colab?: False
Versión de Python : 3.13.12
Versión de PySpark: 4.0.1


**✅ Resultado**

El notebook detecta que corre en un equipo local (`False`), por lo que no reinstala nada y usa el entorno virtual del proyecto: **Python 3.13.12** y **PySpark 4.0.1**. En Colab, la misma celda instala Java 17 y PySpark 4.0.1 automáticamente, de modo que el notebook funciona en ambos entornos sin cambios.

## 1.2 Variables de entorno

**🎯 Qué haremos**

Definir las variables de entorno que Spark lee al arrancar:

| Variable | Para qué sirve |
|---|---|
| `JAVA_HOME` | Carpeta de Java que Spark usa para lanzar la JVM. |
| `SPARK_HOME` | Carpeta con los binarios de Spark (dentro del paquete `pyspark`). |
| `PYSPARK_PYTHON` | Intérprete de Python de los *workers* (procesos que ejecutan las tareas). |
| `PYSPARK_DRIVER_PYTHON` | Intérprete de Python del *driver* (proceso que coordina el trabajo). |

**Decisiones técnicas:**
- `JAVA_HOME` apunta **explícitamente a Java 17**. El equipo local también tiene Java 8, que no es compatible con Spark 4.0; sin esta variable, Spark podría tomar la versión equivocada.
- *Driver* y *workers* usan **el mismo intérprete de Python**. Si las versiones no coinciden, Spark falla al ejecutar funciones de Python sobre los RDDs.

In [4]:
# 1.2 Variables de entorno
import os, glob, platform

if EN_COLAB:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
elif platform.system() == "Windows":
    # Busca la instalación de Java 17 (Eclipse Temurin)
    jdks = sorted(glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*"))
    os.environ["JAVA_HOME"] = jdks[-1]

# Java 17 primero en el PATH, para que tenga prioridad sobre Java 8
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Evidencia: valores configurados
for var in ["JAVA_HOME", "SPARK_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON"]:
    print(f"{var:22s} = {os.environ[var]}")

# Evidencia: versión de Java que usará Spark
java = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
salida = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
print("\nJava que usará Spark:", [l for l in salida.splitlines() if "version" in l][0])

JAVA_HOME              = C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot
SPARK_HOME             = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Lib\site-packages\pyspark
PYSPARK_PYTHON         = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON  = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Scripts\python.exe

Java que usará Spark: openjdk version "17.0.21" 2026-10-20


**✅ Resultado**

Las cuatro variables quedan definidas. `JAVA_HOME` apunta a **Java 17.0.21** (Eclipse Temurin) y no a la versión 8 que también existe en el equipo; la verificación final lo confirma (`openjdk version "17.0.21"`). `SPARK_HOME`, `PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` apuntan al entorno virtual del proyecto, por lo que *driver* y *workers* usan el mismo intérprete.

## 1.3 Creación de la SparkSession y el SparkContext

**🎯 Qué haremos**

Crear los dos puntos de entrada de Spark:
- **`SparkSession` (`spark`)**: punto de entrada a **DataFrames y Spark SQL** (secciones 4 a 8).
- **`SparkContext` (`sc`)**: punto de entrada a los **RDDs** (secciones 2 y 3). Se obtiene con `spark.sparkContext`, de modo que ambos comparten la misma aplicación.

**Decisiones técnicas:**

| Configuración | Valor | Justificación |
|---|---|---|
| `master` | `local[*]` | Ejecuta Spark en este equipo usando **todos los núcleos** disponibles como *workers*. |
| `appName` | `MCDI502_S2_Grupo6` | Identifica la aplicación en la interfaz web de Spark. |
| `spark.sql.shuffle.partitions` | `8` | El valor por defecto (200) está pensado para clústeres. Con datos de este tamaño generaría cientos de tareas casi vacías tras cada `join` o `groupBy`. |
| `setLogLevel("ERROR")` | – | Oculta avisos internos de Spark para que las salidas se lean con claridad. |

In [5]:
# 1.3 SparkSession y SparkContext
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("MCDI502_S2_Grupo6")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

sc = spark.sparkContext          # SparkContext: punto de entrada a los RDDs
sc.setLogLevel("ERROR")

# Evidencia: la sesión está activa
print("Versión de Spark        :", spark.version)
print("Nombre de la aplicación :", sc.appName)
print("Master                  :", sc.master)
print("Paralelismo por defecto :", sc.defaultParallelism, "núcleos")
print("Particiones de shuffle  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Interfaz web de Spark   :", sc.uiWebUrl)

Versión de Spark        : 4.0.1
Nombre de la aplicación : MCDI502_S2_Grupo6
Master                  : local[*]
Paralelismo por defecto : 8 núcleos
Particiones de shuffle  : 8
Interfaz web de Spark   : http://DESKTOP-KD98F3J:4040


**✅ Resultado**

La sesión queda activa con **Spark 4.0.1** en modo `local[*]`. El paralelismo por defecto es **8**, igual al número de núcleos del equipo: Spark puede ejecutar hasta 8 tareas al mismo tiempo, una por núcleo. Las particiones de *shuffle* quedan en 8, alineadas con ese paralelismo. Desde aquí, `spark` se usa para DataFrames y SQL, y `sc` para RDDs. La interfaz web en el puerto 4040 permite monitorear los trabajos mientras se ejecutan.

## 1.4 Ubicación de los datos

**🎯 Qué haremos**

Dejar disponibles los seis archivos originales en una carpeta `data/`:
- **En local:** se usa la carpeta `data/` del repositorio, donde ya están los archivos.
- **En Colab (u otro equipo):** la carpeta no existe, así que se descargan desde el repositorio público del grupo en GitHub.

**Decisión técnica:** la entrega incluye solo el notebook y el video, no los datos. Si el notebook dependiera de subir archivos a mano, no podría ejecutarse "sin intervenciones manuales". Descargarlos desde GitHub hace que cualquier persona pueda ejecutarlo de principio a fin. La ruta se guarda en formato `/` (`as_posix`), que Spark entiende tanto en Windows como en Linux.

In [6]:
# 1.4 Ubicación de los datos
import urllib.request
from pathlib import Path

URL_BASE = "https://raw.githubusercontent.com/jaossandon/mcdi502-s2-spark/main/data/"
ARCHIVOS = ["deportista.csv", "deportista2.csv", "equipo.csv",
            "evento.csv", "resultados.csv", "juegos.json"]

# Busca data/ en la carpeta actual o en las superiores (el notebook está en notebooks/)
carpetas = [Path.cwd(), *Path.cwd().parents]
DATA_DIR = next((p / "data" for p in carpetas if (p / "data" / "deportista.csv").exists()),
                Path("data"))
DATA_DIR.mkdir(exist_ok=True)

for nombre in ARCHIVOS:
    destino = DATA_DIR / nombre
    if not destino.exists():                       # solo descarga si falta
        urllib.request.urlretrieve(URL_BASE + nombre, destino)

DATA = DATA_DIR.resolve().as_posix()               # ruta que usará Spark

# Evidencia: archivos disponibles
print("Carpeta de datos:", DATA, "\n")
print(f"{'Archivo':18s}{'Tamaño (bytes)':>16s}")
for nombre in ARCHIVOS:
    print(f"{nombre:18s}{(DATA_DIR / nombre).stat().st_size:>16,d}")

Carpeta de datos: C:/Users/jorge/Documents/mcdi502-s2-spark/data 

Archivo             Tamaño (bytes)
deportista.csv           2,832,265
deportista2.csv          2,874,875
equipo.csv                  23,606
evento.csv                  35,144
resultados.csv           6,443,911
juegos.json                  5,232


**✅ Resultado**

Los seis archivos están disponibles en la carpeta `data/` del repositorio:

| Archivo | Tamaño | Contenido |
|---|---|---|
| `deportista.csv` | 2,8 MB | deportistas con id 1 al 67.786 |
| `deportista2.csv` | 2,9 MB | deportistas con id 67.787 al 135.571 |
| `equipo.csv` | 24 KB | 1.184 equipos |
| `evento.csv` | 35 KB | 765 eventos |
| `resultados.csv` | 6,4 MB | 271.116 participaciones |
| `juegos.json` | 5 KB | 51 ediciones de los Juegos |

**Cierre de la sección 1:** Java 17 y PySpark 4.0.1 operativos, variables de entorno definidas, `SparkSession` y `SparkContext` activos y datos disponibles. Todo se ejecuta sin intervención manual, tanto en local como en Colab.